В этом ноутбуке я проверил, может ли модель сгенерировать новую json-schema посмотрев на данные. Модель сгенерировала схему, я её сохранил в `new_schema.json`, схема успешно преобразовалась в pydantic BaseModel в файле `model.py`

In [18]:
from utils import llm, ActInfo
from datasets import load_from_disk
from pydantic import BaseModel, Field
import json

In [2]:
dataset = load_from_disk("../data/raw")

In [9]:
refine_prompt = """
Тебе дана json schema, описыващая структуру данных. Проанализируй примеры данных посмотри, что можно улучшить в схеме, если необходимо, добавь описание и примеры. Верни сгенерированную схему в виде текста, и добавь свои рассуждения.

Текущая json schema:
{json_schema}

Примеры данных:
{data_samples}
"""

In [10]:
data_samples = ""

i = 0
for row in dataset["train"]:

    if len(row["text"]) > 1000:
        continue

    data_samples += f"Полный текст:\n{row["text"]}\n"
    data_samples += f"{ActInfo(**row)}\n\n"
    
    i += 1
    if i == 5:
        break

In [11]:
class SchemaGuidedReasoning(BaseModel):
    reasoning: str = Field(description="Какие были внесены изменения и почему")
    refined_json_schema: str = Field(description="Код для создания обновлённой json schema")

In [ ]:
# отработало только с 3 раза
response = llm.with_structured_output(SchemaGuidedReasoning).invoke(refine_prompt.format(json_schema=ActInfo.model_json_schema(), data_samples=data_samples))

In [16]:
print(response.reasoning)

Анализ текущей JSON-схемы и примеров данных выявил несколько возможностей для улучшения:

1. **Уточнение типов данных**: В примерах поле `signatory` всегда содержит инициалы (например, 'Путин В.'), но схема позволяет любые строки. Можно добавить паттерн для валидации формата.

2. **Добавление ограничений для полей**: 
   - `number` в примерах содержит цифры и буквы (например, '669-рп'), но схема разрешает любую строку. Добавим паттерн для соответствия реальным данным.
   - `publication_date` требует формат ISO 8601, но в примерах используется объект datetime. Уточним описание формата.

3. **Улучшение описаний**: 
   - `full_name` в примерах всегда формируется как '{type} {government_agency_name}'. Уточним это в описании.
   - `signatory` имеет специфический формат 'Фамилия И.'. Добавим пример.
   - `type` в примерах всегда начинается с заглавной буквы. Можно добавить ограничение.

4. **Добавление примеров**: Для каждого поля добавим примеры из реальных данных для наглядности.

5. **Уда

In [17]:
print(response.refined_json_schema)

{
  "title": "ActInfo",
  "type": "object",
  "properties": {
    "full_name": {
      "type": "string",
      "minLength": 1,
      "description": "Полное наименование правового акта, сформированное как '{type} {government_agency_name}'. Пример: 'Указ Президента Российской Федерации'",
      "title": "Full Name",
      "example": "Постановление Правительства Российской Федерации"
    },
    "publication_date": {
      "type": "string",
      "format": "date",
      "description": "Дата опубликования документа в формате ISO 8601 (ГГГГ-ММ-ДД). Пример: '2000-07-30'",
      "title": "Publication Date",
      "example": "1997-04-02"
    },
    "number": {
      "type": "string",
      "minLength": 1,
      "pattern": "^[0-9]+[а-яА-ЯёЁ-]*$",
      "description": "Номер документа, содержащий цифры и возможно буквы/дефис. Примеры: '1406', '669-рп', '433-р'",
      "title": "Number",
      "example": "331"
    },
    "title": {
      "type": "string",
      "minLength": 1,
      "description":

In [21]:
json.dump(json.loads(response.refined_json_schema), open("new_schema.json", "w", encoding="utf8"))

In [22]:
! datamodel-codegen --input new_schema.json --input-file-type jsonschema --output-model-type pydantic_v2.BaseModel --output model.py

/home/user/prompt-refining/.venv/lib/python3.13/site-packages/datamodel_code_generator/parser/base.py:2947: FutureWarning: The default formatters (black, isort) will be replaced by ruff in a future version. To prepare for this change, consider using: formatters=[Formatter.RUFF_FORMAT, Formatter.RUFF_CHECK]. Install ruff with: pip install 'datamodel-code-generator[ruff]'. To suppress this warning, specify formatters explicitly.
  code_formatter = CodeFormatter(
